In [22]:
import requests
import pandas as pd
import time
import logging
from pathlib import Path
from tqdm import tqdm

logging.basicConfig(level=logging.INFO, format="%(asctime)s %(levelname)s %(message)s")
log = logging.getLogger(__name__)

API_KEY  = "HDEV-907006be-6b96-44d1-bff6-ade257f217ff"
BASE_URL = "https://api.henrikdev.xyz/valorant"
HEADERS  = {"Authorization": API_KEY}

RATE_SLEEP         = 3.0   # 请求间隔（秒），约 30 req/min
PLAYERS_PER_REGION = 250   # 每地区抽取玩家数（= 每地区比赛数）
REGIONS            = ["na", "eu", "ap", "br", "latam", "kr"]

OUT_DIR = Path("data")
OUT_DIR.mkdir(exist_ok=True)

## 1. API 请求封装

In [23]:
def get(url, params=None, retries=3):
    """带重试的 HTTP GET；遇到限速(429)自动等待"""
    for attempt in range(retries):
        try:
            r = requests.get(url, headers=HEADERS, params=params, timeout=15)
            if r.status_code == 200:
                return r.json()
            elif r.status_code == 429:
                wait = 30 * (attempt + 1)
                log.warning(f"限速(429)，等待 {wait}s …")
                time.sleep(wait)
            else:
                return None
        except Exception as e:
            log.error(f"请求异常: {e}")
            time.sleep(5)
    return None


def fetch_latest_match(puuid, region):
    """拉取玩家最近一场竞技模式比赛，失败返回 None"""
    url  = f"{BASE_URL}/v3/by-puuid/matches/{region}/{puuid}"
    data = get(url, params={"mode": "competitive", "size": 1})
    time.sleep(RATE_SLEEP)
    if not data or str(data.get("status")) != "200":
        return None
    matches = data.get("data", [])
    return matches[0] if matches else None

## 2. 解析单场比赛

In [24]:
def parse_match(match_data):
    """将单场比赛 JSON 解析为每名玩家一行的记录（每场 10 行）"""
    meta = match_data.get("metadata")
    if meta is None:
        return []   # metadata 缺失，跳过这场比赛

    match_id = meta.get("matchid")
    teams    = match_data.get("teams", {})
    team_won = {
        "Red":  teams.get("red",  {}).get("has_won"),
        "Blue": teams.get("blue", {}).get("has_won"),
    }

    rows = []
    for p in match_data.get("players", {}).get("all_players", []):
        s  = p.get("stats",         {})
        ec = p.get("economy",       {})
        bh = p.get("behavior",      {})
        ab = p.get("ability_casts", {})
        total_shots = (s.get("headshots", 0) + s.get("bodyshots", 0) + s.get("legshots", 0)) or 1
        team = p.get("team")

        rows.append({
            # 识别
            "match_id":    match_id,
            "puuid":       p.get("puuid"),
            "name":        p.get("name"),
            "tag":         p.get("tag"),
            "team":        team,
            "region":      meta.get("region"),
            # 控制变量
            "map":            meta.get("map"),
            "patch":          meta.get("game_version"),
            "rounds_played":  meta.get("rounds_played"),
            "game_start":     meta.get("game_start"),
            "rank_tier":      p.get("currenttier"),
            "rank_patched":   p.get("currenttier_patched"),
            "agent":          p.get("character"),
            "player_level":   p.get("level"),
            # 绩效
            "score":           s.get("score"),
            "kills":           s.get("kills"),
            "deaths":          s.get("deaths"),
            "assists":         s.get("assists"),
            "damage_made":     p.get("damage_made"),
            "damage_received": p.get("damage_received"),
            "headshot_rate":   round(s.get("headshots", 0) / total_shots, 4),
            "headshots":       s.get("headshots"),
            "bodyshots":       s.get("bodyshots"),
            "legshots":        s.get("legshots"),
            # 技能
            "c_cast": ab.get("c_cast", 0),
            "q_cast": ab.get("q_cast", 0),
            "e_cast": ab.get("e_cast", 0),
            "x_cast": ab.get("x_cast", 0),
            # 经济
            "economy_spent_overall": ec.get("spent",         {}).get("overall"),
            "economy_spent_avg":     ec.get("spent",         {}).get("average"),
            "loadout_value_overall": ec.get("loadout_value", {}).get("overall"),
            "loadout_value_avg":     ec.get("loadout_value", {}).get("average"),
            # 社会懈怠代理
            "afk_rounds":      bh.get("afk_rounds"),
            "rounds_in_spawn": bh.get("rounds_in_spawn"),
            "ff_outgoing":     bh.get("friendly_fire", {}).get("outgoing"),
            "ff_incoming":     bh.get("friendly_fire", {}).get("incoming"),
            # 结果
            "team_won": team_won.get(team),
        })
    return rows

## 3. 主采集流程

In [25]:
def main(pool_path="players_pool.csv"):
    """
    每个地区随机抽 PLAYERS_PER_REGION 名玩家，
    各拉最近 1 场竞技比赛，最终输出 6×250×10 = 15,000 行。
    """
    # ── Step 1：分层抽样 ──────────────────────────────────
    pool    = pd.read_csv(pool_path)
    pool    = pool[pool["region"].isin(REGIONS)]
    sampled = (
        pool.groupby("region", group_keys=False)
            .apply(lambda g: g.sample(n=min(PLAYERS_PER_REGION, len(g)), random_state=42),
                   include_groups=False)
            .reset_index()  # region 列在 include_groups=False 后会丢失，reset_index 恢复
    )
    # merge 回去补全所有列（包括 region）
    sampled = pool.merge(sampled[["puuid"]], on="puuid").drop_duplicates("puuid").reset_index(drop=True)
    log.info(f"抽样完成：\n{sampled['region'].value_counts().to_string()}")

    # ── Step 2：逐人拉取最近一场比赛并解析 ───────────────
    all_rows = []
    skipped  = 0

    for _, player in tqdm(sampled.iterrows(), total=len(sampled), desc="采集进度", unit="人"):
        match_data = fetch_latest_match(player["puuid"], player["region"])
        if match_data is None:
            log.warning(f"跳过 [{player['region']}] {player.get('name', player['puuid'][:8])}：无数据")
            skipped += 1
            continue
        all_rows.extend(parse_match(match_data))

    # ── Step 3：去重保存 ──────────────────────────────────
    df = pd.DataFrame(all_rows).drop_duplicates(subset=["match_id", "puuid"])

    out_path = OUT_DIR / "match_players.csv"
    df.to_csv(out_path, index=False)

    # 汇报结果
    match_dist = df.drop_duplicates("match_id").groupby("region").size()
    log.info(f"\n各地区比赛数：\n{match_dist}")
    log.info(f"总行数：{len(df)}  |  总比赛数：{df['match_id'].nunique()}  |  跳过：{skipped}")
    log.info(f"✅ 数据已保存 → {out_path}")
    return df

## 4. 执行

In [26]:
df = main("players_pool.csv")
df.to_csv("target_match_dataset.csv", index=False)

2026-04-12 16:50:09,938 INFO 抽样完成：
region
latam    251
kr       250
eu       250
ap       250
na       250
br       249
采集进度: 100%|██████████| 1500/1500 [2:09:50<00:00,  5.19s/人]  
2026-04-12 19:00:01,514 INFO 
各地区比赛数：
region
ap    246
eu    247
kr    245
na    743
dtype: int64
2026-04-12 19:00:01,519 INFO 总行数：14810  |  总比赛数：1481  |  跳过：0
2026-04-12 19:00:01,519 INFO ✅ 数据已保存 → data\match_players.csv


In [28]:
df['region'].value_counts()

region
na    7430
eu    2470
ap    2460
kr    2450
Name: count, dtype: int64